# 04 — Diagnostic Analytics
**Capstone Project — Group 5 — DAMO-699, Summer 2026**

Formal hypothesis tests for RQ1, following the charter's required-relationships table
(section 6.2). Every test reports hypotheses, sample size, assumptions, statistic, p-value,
effect size, and a plain-language interpretation — statistical significance alone is not
enough with 300,000+ rows, since even trivial effects become "significant."


> **Google Colab version.** Only the project-root cell below differs from the VS Code / local version: it mounts Google Drive and points `PROJECT_ROOT` at `My Drive/Colab Notebooks/CAPSTONE` instead of `D:\CAPSTONE PROJECT`. Every other cell, path, and analytical step is identical - `database/`, `Input_Files/`, and `Outputs_Files/` must exist under that Drive folder with the same layout as the local project.


In [7]:
import json
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

pd.set_option("display.max_columns", 120)

from google.colab import drive
drive.mount('/content/drive')

# Colab equivalent of the local D:\CAPSTONE PROJECT root - same folder
# structure (database / Input_Files / Outputs_Files), just mounted from
# Google Drive instead of the Windows filesystem. Update this one line if
# your Drive folder name or location differs.
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/CAPSTONE")
PROCESSED_DIR = PROJECT_ROOT / "database" / "processed"
DICT_DIR = PROJECT_ROOT / "Outputs_Files" / "dictionaries"
OUT_DIR = PROJECT_ROOT / "Outputs_Files" / "04"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / "faa_strikes_clean.csv", low_memory=False)
print(f"Loaded {df.shape[0]:,} rows for diagnostic testing.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 319,107 rows for diagnostic testing.


**What it does:** Loads the modelling-ready cleaned dataset (leakage fields already
removed) that `02_data_cleaning.ipynb` produced.

**Why it matters:** diagnostic tests here feed directly into which features make it into the
Stage 1 damage model in `05` — a relationship that doesn't hold up statistically is a weaker
case for inclusion, though the final feature list still follows the charter's core/extended
split rather than test results alone.

**Result review:** just a row count confirmation.


In [8]:
def chi_square_cramers_v(data, col1, col2, alpha=0.05):
    '''Chi-square test of independence + Cramer's V effect size for two
    categorical columns. Returns a one-row summary dict.'''
    contingency = pd.crosstab(data[col1], data[col2])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan
    return {
        "test": f"{col1} vs {col2}",
        "n": int(n),
        "chi2": round(chi2, 2),
        "dof": dof,
        "p_value": p,
        "significant_at_0.05": p < alpha,
        "cramers_v": round(cramers_v, 4),
        "effect_size_label": (
            "negligible" if cramers_v < 0.1 else
            "small" if cramers_v < 0.3 else
            "medium" if cramers_v < 0.5 else "large"
        ),
    }

relationships = [
    ("WILDLIFE_SIZE_GROUP", "INDICATED_DAMAGE"),
    ("PHASE_OF_FLIGHT", "INDICATED_DAMAGE"),
    ("AC_MASS_GROUP", "INDICATED_DAMAGE"),
    ("WARNED", "INDICATED_DAMAGE"),
    ("SEASON", "INDICATED_DAMAGE"),
    ("FAAREGION", "INDICATED_DAMAGE"),
]
chi_square_results = pd.DataFrame([chi_square_cramers_v(df, a, b) for a, b in relationships])
chi_square_results.to_csv(OUT_DIR / "04_chi_square_cramers_v_results.csv", index=False)
chi_square_results

,test,n,chi2,dof,p_value,significant_at_0.05,cramers_v,effect_size_label
0,WILDLIFE_SIZE_GROUP vs INDICATED_DAMAGE,319107,33302.66,3,0.000000e+00,True,0.3231,medium
1,PHASE_OF_FLIGHT vs INDICATED_DAMAGE,319107,13854.72,11,0.000000e+00,True,0.2084,small
2,AC_MASS_GROUP vs INDICATED_DAMAGE,319107,21224.31,3,0.000000e+00,True,0.2579,small
3,WARNED vs INDICATED_DAMAGE,319107,4093.99,2,0.000000e+00,True,0.1133,small
4,SEASON vs INDICATED_DAMAGE,319107,2249.76,3,0.000000e+00,True,0.0840,negligible
5,FAAREGION vs INDICATED_DAMAGE,278004,391.81,9,7.670904e-79,True,0.0375,negligible


**What it does:** Runs a chi-square test of independence plus Cramér's V effect size for
every categorical-vs-damage relationship the charter lists as required, in one loop.

**Why it matters:** with 300,000+ rows, almost every relationship will hit p < 0.05 whether it
matters practically or not. Cramér's V is what actually tells us whether an association is
worth building a feature around — the charter is explicit that effect size must be reported
alongside significance, not instead of it.

**Result review:** expect every p-value here to be far below 0.05 (sample size alone almost
guarantees that). What matters is the `cramers_v` column — `WILDLIFE_SIZE_GROUP` and
`PHASE_OF_FLIGHT` should show the largest effect sizes among these; if any relationship comes
back "negligible" despite a tiny p-value, that's the sample-size-inflates-significance point
in practice, and it argues against leaning heavily on that feature alone.


In [9]:
# H3: larger wildlife and more animals struck associate with higher damage
# probability and severity - trend test on NUM_STRUCK via Mann-Whitney U
# (damaged vs non-damaged groups), since NUM_STRUCK is not normally distributed.

import pandas as pd
import numpy as np
from scipy import stats

# Convert NUM_STRUCK to numeric
df["NUM_STRUCK"] = pd.to_numeric(df["NUM_STRUCK"], errors="coerce")

# Remove missing values
damaged_struck = df.loc[df["INDICATED_DAMAGE"] == 1, "NUM_STRUCK"].dropna()
nondamaged_struck = df.loc[df["INDICATED_DAMAGE"] == 0, "NUM_STRUCK"].dropna()

# Mann-Whitney U test
u_stat, p_value = stats.mannwhitneyu(
    damaged_struck,
    nondamaged_struck,
    alternative="two-sided"
)

# Rank-biserial correlation
n1, n2 = len(damaged_struck), len(nondamaged_struck)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

print("H3 (NUM_STRUCK): Mann-Whitney U test, damaged vs non-damaged strikes")
print(f"n damaged = {n1}, n non-damaged = {n2}")
print(f"U statistic = {u_stat:.1f}, p-value = {p_value:.2e}")
print(f"Rank-biserial correlation (effect size) = {rank_biserial:.4f}")
print(f"Median NUM_STRUCK - damaged: {damaged_struck.median()}, non-damaged: {nondamaged_struck.median()}")


H3 (NUM_STRUCK): Mann-Whitney U test, damaged vs non-damaged strikes
n damaged = 20782, n non-damaged = 297644
U statistic = 3390731570.0, p-value = 0.00e+00
Rank-biserial correlation (effect size) = -0.0963
Median NUM_STRUCK - damaged: 1.0, non-damaged: 1.0


**What it does:** Compares the number of animals struck between damaging and non-damaging
incidents using a Mann-Whitney U test (a rank-based test that doesn't assume a normal
distribution, appropriate here since `NUM_STRUCK` is heavily right-skewed).

**Why it matters:** this is the direct test for hypothesis H3 from the proposal — more
animals struck should mean a higher chance of damage. A parametric t-test would be the wrong
tool here since the data isn't remotely normal (most strikes involve 1 animal).

**Result review:** expect a positive rank-biserial correlation and a higher median
`NUM_STRUCK` among damaged strikes, supporting H3. The p-value will almost certainly be tiny
given the sample size — the rank-biserial value is what indicates whether that difference is
actually large enough to matter in practice.


In [10]:
height_damaged = df.loc[df["INDICATED_DAMAGE"] == 1, "HEIGHT"].dropna()
height_nondamaged = df.loc[df["INDICATED_DAMAGE"] == 0, "HEIGHT"].dropna()
speed_damaged = df.loc[df["INDICATED_DAMAGE"] == 1, "SPEED"].dropna()
speed_nondamaged = df.loc[df["INDICATED_DAMAGE"] == 0, "SPEED"].dropna()

height_u, height_p = stats.mannwhitneyu(height_damaged, height_nondamaged, alternative="two-sided")
speed_u, speed_p = stats.mannwhitneyu(speed_damaged, speed_nondamaged, alternative="two-sided")

print(f"HEIGHT - Mann-Whitney U={height_u:.1f}, p={height_p:.2e}, "
      f"median damaged={height_damaged.median()}, median non-damaged={height_nondamaged.median()}")
print(f"SPEED  - Mann-Whitney U={speed_u:.1f}, p={speed_p:.2e}, "
      f"median damaged={speed_damaged.median()}, median non-damaged={speed_nondamaged.median()}")

HEIGHT - Mann-Whitney U=1388059639.0, p=0.00e+00, median damaged=400.0, median non-damaged=25.0
SPEED  - Mann-Whitney U=495863790.5, p=4.75e-19, median damaged=135.0, median non-damaged=140.0


**What it does:** Runs the same Mann-Whitney U comparison for `HEIGHT` and `SPEED` between
damaging and non-damaging strikes.

**Why it matters:** these are two of the required numeric-vs-damage relationships from the
charter's table, and both feed the extended feature set in modelling.

**Result review:** expect damaging strikes to skew toward lower altitude (most strikes happen
near the ground during take-off/landing regardless of damage, but damage often correlates with
specific flight phases more than raw height). Speed differences are worth checking against the
phase-of-flight breakdown from `03`, since speed and phase are closely linked.


In [11]:
# Multivariable logistic regression (interpretable baseline, odds ratios).
# Uses CORE_FEATURES only here - the full comparison against extended features happens in 05.
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path
# Correct path to feature_lists.json
feature_path = DICT_DIR / "feature_lists.json"  # use the DICT_DIR already set in the first cell

with open(feature_path, "r") as f:
    feature_lists = json.load(f)

numeric_like = {"NUM_STRUCK", "NUM_ENGS", "MONTH_SIN", "MONTH_COS"}

# Use the filtered feature list here
formula_features = [f for f in feature_lists["CORE_FEATURES"] if f not in ("MONTH_SIN", "MONTH_COS")]

model_df = df[formula_features + ["INDICATED_DAMAGE"]].copy()

# Convert numeric-like columns safely
for col in numeric_like.intersection(model_df.columns):
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

# Drop constant columns that can cause singularity
constant_cols = [c for c in formula_features if model_df[c].nunique(dropna=True) <= 1]
if constant_cols:
    print("Dropping constant columns:", constant_cols)
    model_df = model_df.drop(columns=constant_cols)
    formula_features = [c for c in formula_features if c not in constant_cols]

# Build Patsy-style formula
terms = [
    f"C({f})" if f not in numeric_like else f
    for f in formula_features
]
formula = "INDICATED_DAMAGE ~ " + " + ".join(terms)

# Drop missing rows only for the variables used in the model
model_df = model_df.dropna(subset=["INDICATED_DAMAGE"] + formula_features).copy()

# Fit logistic regression
try:
    logit_model = smf.logit(formula, data=model_df).fit(disp=0)
except np.linalg.LinAlgError:
    print("Standard logit failed because the design matrix is singular.")
    print("Try removing a redundant predictor or using regularization.")
    logit_model = smf.logit(formula, data=model_df).fit_regularized(alpha=0.1, L1_wt=0.0)

# Odds ratios
odds_ratios = np.exp(logit_model.params)

# Confidence intervals if available
try:
    conf_int = np.exp(logit_model.conf_int())
    odds_ratio_table = pd.DataFrame({
        "odds_ratio": odds_ratios,
        "ci_low": conf_int[0],
        "ci_high": conf_int[1],
        "p_value": getattr(logit_model, "pvalues", np.nan)
    }).round(4)
except Exception:
    odds_ratio_table = pd.DataFrame({
        "odds_ratio": odds_ratios
    }).round(4)

odds_ratio_table.to_csv(OUT_DIR / "04_logistic_regression_odds_ratios.csv")

print(logit_model.summary())



/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))


Standard logit failed because the design matrix is singular.
Try removing a redundant predictor or using regularization.
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.23316559370627685
            Iterations: 651
            Function evaluations: 653
            Gradient evaluations: 651


/usr/local/lib/python3.12/dist-packages/statsmodels/base/l1_solvers_common.py:71: ConvergenceWarning: QC check did not pass for 12 out of 29 parameters
Try increasing solver accuracy or number of iterations, decreasing alpha, or switch solvers
  warnings.warn(message, ConvergenceWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/base/l1_solvers_common.py:144: ConvergenceWarning: Could not trim params automatically due to failed QC check. Trimming using trim_mode == 'size' will still work.
  warnings.warn(msg, ConvergenceWarning)


                           Logit Regression Results                           
Dep. Variable:       INDICATED_DAMAGE   No. Observations:               186497
Model:                          Logit   Df Residuals:                   186468
Method:                           MLE   Df Model:                           28
Date:                Wed, 29 Jul 2026   Pseudo R-squ.:                  0.2130
Time:                        14:50:59   Log-Likelihood:                -43483.
converged:                       True   LL-Null:                       -55251.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -0.4247      0.109     -3.885      0.000      -0.639      -0.210
C(SEASON)[T.Spring]                  -0.0129      0.024     -0

**What it does:** Fits a multivariable logistic regression of `INDICATED_DAMAGE` on the
core feature set, converts coefficients to odds ratios with confidence intervals, and saves
the table.

**Why it matters:** this is the interpretable baseline the charter requires before any
black-box model — it answers whether wildlife size, flight phase, aircraft mass, and the
other core features hold up as damage predictors once they're all considered together,
instead of one at a time like the chi-square tests above.

**Result review:** an odds ratio above 1 means that category raises the odds of damage versus
the reference category; below 1 lowers it. Check whether `WILDLIFE_SIZE_GROUP=Large` and
higher `NUM_STRUCK` show odds ratios clearly above 1, and whether their confidence intervals
exclude 1 (statistically distinguishable from no effect). This model is the one carried
forward as the "logistic regression baseline" in `05_damage_modelling.ipynb`.


In [12]:
# Priority interactions (charter section 17): wildlife size x aircraft mass,
# wildlife size x flight phase, speed x flight phase.
interaction_formula = (
    "INDICATED_DAMAGE ~ C(WILDLIFE_SIZE_GROUP) * C(AC_MASS_GROUP) "
    "+ C(WILDLIFE_SIZE_GROUP) * C(PHASE_OF_FLIGHT)"
)
interaction_df = df[["WILDLIFE_SIZE_GROUP", "AC_MASS_GROUP", "PHASE_OF_FLIGHT", "INDICATED_DAMAGE"]].dropna()
interaction_model = smf.logit(interaction_formula, data=interaction_df).fit(disp=0)

interaction_terms = interaction_model.params[interaction_model.params.index.str.contains(":")]
interaction_pvalues = interaction_model.pvalues[interaction_terms.index]
interaction_summary = pd.DataFrame({
    "coefficient": interaction_terms.round(4),
    "odds_ratio": np.exp(interaction_terms).round(4),
    "p_value": interaction_pvalues,
})
interaction_summary.to_csv(OUT_DIR / "04_interaction_terms.csv")

print(f"Interaction terms tested: {len(interaction_summary)}")
print(f"Significant at 0.05: {(interaction_summary['p_value'] < 0.05).sum()}")
interaction_summary.sort_values("p_value").head(10)

Interaction terms tested: 42
Significant at 0.05: 19


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,coefficient,odds_ratio,p_value
C(WILDLIFE_SIZE_GROUP)[T.Small]:C(PHASE_OF_FLIGHT)[T.Take-off Run],0.6672,1.9488,7.701796e-19
C(WILDLIFE_SIZE_GROUP)[T.Unknown]:C(PHASE_OF_FLIGHT)[T.Unknown],1.3712,3.9401,2.385612e-18
C(WILDLIFE_SIZE_GROUP)[T.Unknown]:C(AC_MASS_GROUP)[T.Unknown],2.4787,11.9262,1.034055e-14
C(WILDLIFE_SIZE_GROUP)[T.Small]:C(PHASE_OF_FLIGHT)[T.Unknown],0.9219,2.5142,3.070013e-10
C(WILDLIFE_SIZE_GROUP)[T.Medium]:C(PHASE_OF_FLIGHT)[T.Landing Roll],-0.4528,0.6358,4.172505e-10
C(WILDLIFE_SIZE_GROUP)[T.Medium]:C(PHASE_OF_FLIGHT)[T.Unknown],0.8699,2.3867,1.011565e-09
C(WILDLIFE_SIZE_GROUP)[T.Medium]:C(PHASE_OF_FLIGHT)[T.Take-off Run],0.4165,1.5166,1.117463e-09
C(WILDLIFE_SIZE_GROUP)[T.Small]:C(AC_MASS_GROUP)[T.Light],-0.3417,0.7106,4.516156e-08
C(WILDLIFE_SIZE_GROUP)[T.Small]:C(PHASE_OF_FLIGHT)[T.Landing Roll],-0.3263,0.7216,5.213824e-05
C(WILDLIFE_SIZE_GROUP)[T.Small]:C(AC_MASS_GROUP)[T.Unknown],-0.9957,0.3695,2.086882e-04


**What it does:** Adds interaction terms (wildlife size × aircraft mass, wildlife size ×
flight phase) to the logistic model and pulls out just the interaction coefficients.

**Why it matters:** these are the two priority interactions the charter specifically calls
out — the idea that wildlife size might matter more for light aircraft than heavy ones, for
example, wouldn't show up in a model that only has main effects.

**Result review:** a significant interaction term with a p-value under 0.05 means the effect
of wildlife size genuinely depends on aircraft mass (or flight phase) rather than being the
same across every category. These findings become priority candidates for the SHAP
interaction analysis in `09_explainability.ipynb`.


## Summary — Association, Not Causation

Every result above is an **association**, not a causal effect. The FAA database is
observational: it records what was reported, not a controlled experiment. Statements like
"large wildlife causes more damage" go beyond what these tests can support — the accurate
version is "large wildlife is associated with a higher damage rate, controlling for aircraft
mass and flight phase, at effect size X." This distinction is repeated deliberately, because
the charter requires every analytical narrative to keep frequency, association, adjusted
association, prediction, and causation clearly separated (Week 3 checkpoint).

**Limitations:** results reflect *reported* strikes only, so any factor that influences
whether a strike gets reported in the first place (not just whether it causes damage) can
distort these associations. Regional and airport differences may reflect reporting culture as
much as ecological risk.
